RunnablePassthrough

유용한 시나리오
- 데이터를 변환하거나 수정할 필요가 없는 경우
- 파이프라인의 특정 단계를 건너뛰어야 하는 경우
- 디버깅이나 테스트 목적으로 데이터 흐름을 모니터링 해야하는 경우

In [ ]:
from dotenv import load_dotenv

from langchain_teddynote import logging
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser

In [ ]:
load_dotenv()

In [ ]:
logging.langsmith("langchain-LCEL-Advanced")

In [ ]:
runnable = RunnableParallel(
    # passed: 전달된 입력을 그대로 반환하는 Runnable 설정
    passed=RunnablePassthrough(), 
    # extra: 계산 결과를 정의한 key에 할당하는 작업 정의
    extra=RunnablePassthrough.assign(mult=lambda x: x["num"] * 3), 
    # modified: 
    modified=lambda x: x["num"] + 1
)

In [ ]:
runnable.invoke({"num": 1})

In [ ]:
r = RunnablePassthrough.assign(mult=lambda x: x["num"] * 3)

In [ ]:
r.invoke({"num": 1})

검색기 예제

In [ ]:
embedding = OpenAIEmbeddings()

In [ ]:
vectorstore = FAISS.from_texts(
    [
        "테디는 랭체인 주식회사에서 근무를 하였습니다.",
        "셜리는 테디와 같은 회사에서 근무하였습니다.",
        "테디의 직업은 개발자입니다.",
        "셜리의 직업은 디자이너입니다.",
    ], 
    embedding=embedding
)

In [ ]:
retriever = vectorstore.as_retriever()  # 벡터DB를 retriever로 사용

In [ ]:
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)  # 프롬프트 생성

In [ ]:
model = ChatOpenAI(model_name="gpt-4o-mini")

In [ ]:
# 문서를 포맷팅하는 함수
def format_docs(docs):
    return "\n".join([doc.page_content for doc in docs])

In [ ]:
retrieval_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()} 
    | prompt 
    | model 
    | StrOutputParser()
)

In [ ]:
retrieval_chain.invoke("테디의 직업은 무엇입니까?")

In [ ]:
retrieval_chain.invoke("셜리의 직업은 무엇입니까?")